# Problema de la mochila

bela gallardo ing fernandez

# Problema del agente viajero

Dado un grafo no dirigido y con pesos $G = (V, A)$ encontrar un ciclo simple de costo mínimo que pase por todos los nodos.

Características:
- Es un problema NP (No polinomial), pero necesitamos una solución eficiente.
Posibilidades:
1. **Los nodos son los candidatos**. Empezar en un nodo cualquiera. En cada paso moverse al nodo no visitado más próximo al último nodo seleccionado.


# Recocido simulado (simulated annealing)

- Hacer este problema para que podamos hacer que el intervalo sea de cierta profundidad de bits (no solo la normal)
- Hacer un análisis de sensibilización de parámetros.
Definir la función de temperatura para este algoritmo.

Consta de 2 cosas:
- Exploración (buscamos en todo el espacio)
- Explotación (si encontramos algo muy bueno, nos quedamos ahí)

Cómo escogemos la $T_{max}$?
Entre mas grandes sean los cambios de $\Delta E$ para la función objetivo, con esto podemos determinar le valor de la $T_{max}$, así que debemos ver cual es cambio y sacar un promedio y poner un valor parecido (en cantidad) a los cambios para evitar la convergencia prematura del algoritmo.


Definir el intervalo de búsqueda con cierta cantidad de bits, y además normalizar estos valores.

In [28]:
import random
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [69]:
def decodificar_binario(bits, a, b):
    """
    Convierte una lista de bits a un valor real en el intervalo [a, b].
    """
    n = len(bits)
    # Convertir lista de bits a string y luego a entero decimal
    decimal = int("".join(str(bit) for bit in bits), 2)
    # Interpolar en el intervalo [a, b] (normalizar)
    return a + decimal * (b - a) / ((2**n) - 1)

def de_jong_1(x):
    """
    Función objetivo: Polinomio de De Jong 1 (Esfera) en 1D.
    f(x) = x^2
    """
    return -x**2 + 34*x + 546

In [70]:
def Recocido_Binario(n_bits, a, b, func, minimizar=True, 
                     T_max=100.0, epsilon=0.01, tol=1e-4, 
                     tipo_temp='asintotica', alpha=0.1, iter_por_T=10):
    
    # Generar solución inicial aleatoria en formato binario
    s_actual_bits = [random.choice([0, 1]) for _ in range(n_bits)] # Creamos aleatoriamente un punto de inicio en binario
    s_actual_x = decodificar_binario(s_actual_bits, a, b)
    
    mejor_bits = list(s_actual_bits)
    mejor_x = s_actual_x
    
    T = T_max
    k = 1 # Contador de épocas para la temperatura asintótica
    
    # Lista para almacenar los registros de la tabla
    registros_tabla = []

    # Historial para graficar
    historial_x = [s_actual_x]
    historial_y = [func(s_actual_x)]
    
    s_anterior_x = s_actual_x + tol * 10 # Evitamos que no se cumpla la tolerancia en la iteración 1
    
    # T > epsilon (temperatura minima) Y la diferencia entre soluciones es mayor a la Tolerancia
    while T > epsilon and abs(s_actual_x - s_anterior_x) >= tol:
        
        s_anterior_x = s_actual_x # Guardamos la solución de la época anterior
        
        # Evaluamos para una temperatura concreta que luego haremos descender
        for _ in range(iter_por_T):
            # Generar vecino cercano
            s_vecino_bits = list(s_actual_bits)
            idx = random.randint(0, n_bits - 1)
            s_vecino_bits[idx] = 1 - s_vecino_bits[idx] # Cambia 0 a 1, o 1 a 0
            
            s_vecino_x = decodificar_binario(s_vecino_bits, a, b)
            
            # Calcular delta E
            delta_E = func(s_vecino_x) - func(s_actual_x)
            
            # Si queremos maximizar, preguntamos y solo cambiamos el signo para evaluar el cambio
            if not minimizar:
                delta_E = -delta_E 
            
            # Como solo podemos aceptar si es deltaE <= 0 o la ec de Woltzman lo permite
            aceptado = False
            # Criterio de aceptación
            if delta_E <= 0:
                aceptado = True
                # Evaluar si es el mejor histórico
                if (minimizar and func(s_vecino_x) < func(mejor_x)) or \
                   (not minimizar and func(s_vecino_x) > func(mejor_x)): ## Aqui preguntamos si queremos minimo o máximo
                    mejor_bits = list(s_vecino_bits)
                    mejor_x = s_vecino_x
            else:
                probabilidad = math.exp(-delta_E / T)
                if random.random() < probabilidad:
                    aceptado = True
            
            # Si se acepta, el vecino se vuelve la solución actual
            if aceptado:
                s_actual_bits = list(s_vecino_bits)
                s_actual_x = s_vecino_x
                
        

        # Guardar el registro para la tabla ANTES de actualizar la solución
        str_vecino = "".join(str(b) for b in s_vecino_bits)
        registros_tabla.append({
            "T": round(T, 1) if not T.is_integer() else int(T),
            # "Move": s_vecino_x
            "Solution": str_vecino,
            "x": s_actual_x,
            "Δf": delta_E,
            "Move?": "Yes" if aceptado else "No",
            "New Neighbor-Solution": str_vecino if aceptado else "".join(str(b) for b in s_actual_bits),
            'y': func(s_actual_x)
        })

        # Guardar solución de esta época para la gráfica
        historial_x.append(s_actual_x)
        historial_y.append(func(s_actual_x))
        
        # Actualización de Temperatura
        if tipo_temp == 'lineal':
            # T = T - alpha
            T = T - alpha
        elif tipo_temp == 'asintotica':
            T = T_max / (1 + alpha * k)
            
        k += 1 # Incrementar época

    return pd.DataFrame(registros_tabla)


In [75]:
a = -10
b = 10
tipo_temp = 'asintotica' # 'lineal' o 'asintotica'
Datos = Recocido_Binario(
    n_bits = 10,            # Longitud de la cadena binaria
    a = a,              # Límite inferior (Estándar para De Jong)
    b = b,               # Límite superior
    func = de_jong_1,       # Función a optimizar
    minimizar = False,       # True para Minimizar, False para Maximizar
    T_max = 600.0,          # Temperatura inicial
    epsilon = 0.001,        # Criterio de parada por enfriamiento (épsilon)
    tol = 1e-4,             # Criterio de parada por tolerancia de cambio
    tipo_temp = tipo_temp, # 'lineal' o 'asintotica'
    alpha = 0.5,            # Tasa de decaimiento
    iter_por_T = 10         # Vecinos evaluados por nivel de temperatura
)

print(Datos.to_string(index=False)) 


    T   Solution         x         Δf Move? New Neighbor-Solution          y
600.0 1101111011  7.419355 -25.540582   Yes            1101111011 743.211238
400.0 0001110001 -7.790811   1.937147   Yes            0001110001 220.415673
300.0 1011111101  4.956012  -3.791887   Yes            1011111101 689.942347
240.0 1100010100  5.405670  27.448604   Yes            1100010100 700.571503
200.0 0100000100  5.092864 338.571105    No            1100000100 693.220115
171.4 1100111100  6.187683  47.852205   Yes            1100111100 718.093807
150.0 1110111010  8.651026 -48.047899   Yes            1110111010 765.294640
133.3 1100111000  6.109482  25.687353   Yes            1100111000 716.396616
120.0 1101010001  6.598240   0.811903   Yes            1101010001 726.803399
109.1 1011010010  4.115347   0.503417   Yes            1011010010 668.985718
100.0 0111111000 -0.146628   1.339361   Yes            0111111000 540.993163
 92.3 1101001110  6.539589  12.696906   Yes            1101001110 725.579811

In [77]:
from matplotlib.animation import FuncAnimation, PillowWriter

# Fondo estático
fig, ax = plt.subplots(figsize=(7, 4))

# Función objetivo
x_vals = np.linspace(a, b, 500)
y_vals = [de_jong_1(val) for val in x_vals]
ax.plot(x_vals, y_vals, color='black', alpha=0.3, label='Función Objetivo F(x)')

# Marcar el mejor punto encontrado (esto tampoco cambia)
ax.scatter(Datos['x'].iloc[-1], Datos['y'].iloc[-1], color='gold', 
           edgecolors='black', s=200, zorder=5, marker='*', 
           label='Mejor Solución Encontrada')

# Inicializar el punto móvil (vacío por ahora)
# Usamos ax.plot en lugar de scatter porque es más fácil de actualizar en animaciones
punto_actual, = ax.plot([], [], 'bo', markersize=8, zorder=5, label='Punto actual')

# Configuraciones de diseño
ax.set_title(f'Optimización por Recocido Simulado Binario\nEnfriamiento: {tipo_temp.capitalize()} | Óptimo en x = {Datos['x'].iloc[-1]:.4f}')
ax.set_xlabel('Valor decodificado (x)')
ax.set_ylabel('f(x)')
ax.legend()
ax.grid(True)

# 2. Definir la función de actualización
def actualizar(frame):
    """
    Esta función se llama en cada cuadro (frame).
    Actualiza las coordenadas (x, y) y el color del punto.
    """
    # Si estamos en los últimos 10 puntos, cambiamos el color a verde
    if frame >= len(Datos['x']) - 10:
        punto_actual.set_color('green')
    else:
        # Nos aseguramos de que el resto de los puntos se mantengan azules
        punto_actual.set_color('blue')
        
    punto_actual.set_data([Datos['x'].iloc[frame]], [Datos['y'].iloc[frame]])
    return punto_actual,

# 3. Crear la animación
# frames = cantidad total de puntos en tu historial 'x'
anim = FuncAnimation(fig, actualizar, frames=len(Datos['x']), blit=True)

# 4. Guardar la animación como GIF
print("Generando el archivo GIF... (Esto puede tomar unos segundos)")

# fps = frames per second (cuadros por segundo). A mayor número, más rápida la animación.
anim.save('trayectoria_recocido.gif', writer=PillowWriter(fps=10)) 

print("¡GIF guardado exitosamente como 'trayectoria_recocido.gif'!")

# Opcional: Cerrar la figura para liberar memoria si estás en un script largo
plt.close(fig)

Generando el archivo GIF... (Esto puede tomar unos segundos)
¡GIF guardado exitosamente como 'trayectoria_recocido.gif'!
